# 03_select_reference_model.ipynb

Chọn reference model từ ba checkpoint đã train. Đo số tham số, dung lượng và độ trễ, đánh giá lại trên validation, rồi thử năm suy giảm ảnh cố định. Không train lại. Notebook chạy trên Kaggle với GPU.

Cell môi trường kéo nhánh `reference-model-selection` và gắn ảnh từ `cthng123/durian-leaf-disease-dataset`. Tập test không mở.

## 1. Môi trường

Internet để On. Accelerator chọn GPU. Dataset input gồm `cthng123/durian-leaf-disease-dataset` và dataset chứa ba file checkpoint `resnet50_seed42.pt`, `convnext_tiny_seed42.pt`, `mobilenetv3_small_seed42.pt`.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "reference-model-selection"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

Cloning into '/kaggle/working/durian-leaf-disease-robustness'...


Nhánh: reference-model-selection
Commit: 687e6d6
Ảnh: /kaggle/working/durian-leaf-disease-robustness/datasets/durian-ldd/DLD_FinalDataset_224_spit


## 2. Checkpoint

Tìm ba file checkpoint trong `/kaggle/input` và gom vào một thư mục.

In [2]:
from src.evaluation.reference import collect_checkpoint

CHECKPOINT_NAMES = ["resnet50_seed42.pt", "convnext_tiny_seed42.pt", "mobilenetv3_small_seed42.pt"]
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")

for name in CHECKPOINT_NAMES:
    path = collect_checkpoint("/kaggle/input", name, CHECKPOINT_DIR)
    print(f"{name}: {path.stat().st_size / 1024**2:.1f} MB")

resnet50_seed42.pt: 90.0 MB
convnext_tiny_seed42.pt: 106.2 MB
mobilenetv3_small_seed42.pt: 5.9 MB


## 3. Đo và chọn

Mỗi model được đánh giá sạch và dưới năm suy giảm trên validation, rồi đo độ trễ batch 1 và batch 32. Quy tắc chọn nằm trong `configs/reference_selection.yaml`.

In [3]:
from src.evaluation.reference import main

record = main("configs/reference_selection.yaml", CHECKPOINT_DIR)

resnet50: clean 0.9720  corrupted 0.7073  5.491 ms/ảnh
convnext_tiny: clean 0.9775  corrupted 0.7709  5.975 ms/ảnh
mobilenetv3_small: clean 0.9292  corrupted 0.6287  5.294 ms/ảnh
Reference model: convnext_tiny (macro F1 sạch ngang nhau, mức giảm dưới suy giảm thấp nhất)


## 4. Kết quả

Cột `matches_recorded` cho biết checkpoint tái hiện đúng macro F1 đã ghi lúc train.

In [4]:
import pandas as pd

models = record["models"]
display(pd.DataFrame([
    {
        "model": name,
        "parameters": entry["model"]["parameters"],
        "size_mb": entry["model"]["size_mb"],
        "ms_per_image_batch1": entry["latency"][0]["ms_per_image"],
        "images_per_second_batch32": entry["latency"][1]["images_per_second"],
    }
    for name, entry in models.items()
]))
display(pd.DataFrame([
    {
        "model": name,
        "clean_macro_f1": entry["clean"]["macro_f1"],
        "recorded_macro_f1": entry["clean"]["recorded_macro_f1"],
        "matches_recorded": entry["clean"]["matches_recorded"],
        **entry["corrupted"]["macro_f1"],
        "mean_corrupted": entry["corrupted"]["mean_macro_f1"],
        "degradation": entry["corrupted"]["degradation"],
    }
    for name, entry in models.items()
]))
print(f"Reference model: {record['selection']['reference']}")
print(f"Lý do: {record['selection']['reason']}")
print("Kết quả ghi tại reports/baselines/reference_selection.json")

,model,parameters,size_mb,ms_per_image_batch1,images_per_second_batch32
0,resnet50,23518277,89.72,5.491,382.9
1,convnext_tiny,27823973,106.14,5.975,270.1
2,mobilenetv3_small,1522981,5.81,5.294,4318.1


,model,clean_macro_f1,recorded_macro_f1,matches_recorded,brightness_dark,brightness_bright,contrast_low,blur,noise,mean_corrupted,degradation
0,resnet50,0.972039,0.972039,True,0.908882,0.920466,0.734537,0.404720,0.567839,0.707289,0.264750
1,convnext_tiny,0.977461,0.977461,True,0.892268,0.850790,0.876149,0.576963,0.658203,0.770875,0.206586
2,mobilenetv3_small,0.929203,0.929203,True,0.777947,0.861031,0.781065,0.398387,0.325315,0.628749,0.300454


Reference model: convnext_tiny
Lý do: macro F1 sạch ngang nhau, mức giảm dưới suy giảm thấp nhất
Kết quả ghi tại reports/baselines/reference_selection.json
